# Modelling — Credit Risk & Loan Default

Anomaly detection, feature engineering and selection, tuning, evaluation, explainability and
error analysis. Every function called here lives in the `credit_risk` package, so the same code
runs in this notebook, in the CLI and inside the container.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from credit_risk import silence_library_warnings
from credit_risk.anomaly import handling
from credit_risk.config import CONFIG
from credit_risk.data import RAW_CSV, TARGET_COLUMN, read_frame
from credit_risk.error_analysis import (
    classify_predictions,
    confident_mistakes,
    error_profile,
    error_rate_by_segment,
    error_summary,
    plot_error_overview,
    plot_error_rate_by_segment,
)
from credit_risk.evaluation import (
    cross_validated_score,
    optimal_threshold,
    out_of_fold_probabilities,
    plot_calibration_curve,
    plot_confusion_matrix,
    plot_learning_curve,
    plot_model_comparison,
    plot_roc_and_pr_curves,
    plot_threshold_cost,
)
from credit_risk.explain import (
    decision_reasons,
    explain_model,
    global_importance,
    local_contributions,
    plot_beeswarm,
    plot_dependence,
    plot_importance_bar,
    top_features,
)
from credit_risk.features.engineering import ENGINEERED_NUMERIC, engineer_features
from credit_risk.features.selection import (
    candidate_features,
    plot_selection_curve,
    rank_features,
    select_features,
    split_feature_types,
)
from credit_risk.pipeline import MODEL_BUILDERS, build_model
from credit_risk.train import calibrate_model, score_model, split_frame
from credit_risk.tuning import plot_param_importances, plot_tuning_history, tune_model

silence_library_warnings()

applicants = read_frame(RAW_CSV)
print(f"{len(applicants)} applicants, approval rate {applicants[TARGET_COLUMN].mean():.1%}")

## 1. Anomaly detection

Four lenses on "unusual": a domain rule (impossible values), univariate statistics (IQR,
z-score), and two multivariate detectors (Isolation Forest, One-Class SVM) that can flag a row
whose *combination* of values is odd even when no single value is.

In [ ]:
flags = handling.all_outlier_flags(applicants)
handling.summarize_outliers(flags, applicants)

In [ ]:
handling.detector_agreement(flags)

### Which rows are flagged, and by which methods

In [ ]:
consensus = handling.consensus_outliers(flags)
handling.list_outliers(applicants, flags, consensus)

### Do the outliers deserve removal?

Removal is only ever applied to *training* rows — deleting holdout rows would flatter the
score rather than improve the model.

In [ ]:
train_frame, holdout_frame = split_frame(engineer_features(applicants))
cleaned, removed = handling.clean_training_frame(train_frame)

features = candidate_features()
numeric, categorical = split_feature_types(features)
with_outliers = cross_validated_score(
    train_frame[features], train_frame[TARGET_COLUMN], "lightgbm", numeric, categorical
)
without_outliers = cross_validated_score(
    cleaned[features], cleaned[TARGET_COLUMN], "lightgbm", numeric, categorical
)

print(f"consensus outliers in the training rows: {int(removed.sum())}")
print(f"cross-validated PR-AUC with outliers   : {with_outliers['mean']:.4f}")
print(f"cross-validated PR-AUC without them    : {without_outliers['mean']:.4f}")
print(f"change: {without_outliers['mean'] - with_outliers['mean']:+.4f}")

## 2. Feature engineering

Missingness indicators, domain interactions, ratios and credit bands. The transformer is
stateless — nothing is learned from the data — so applying it before the split cannot leak.

In [ ]:
engineered = engineer_features(applicants)
uplift = pd.DataFrame(
    {
        "corr_with_target": engineered[ENGINEERED_NUMERIC + ["CreditScore", "Income"]]
        .corrwith(engineered[TARGET_COLUMN])
        .round(3)
    }
).sort_values("corr_with_target", key=abs, ascending=False)
uplift

In [ ]:
banded = engineered.groupby("credit_band", dropna=False, observed=True)[TARGET_COLUMN].agg(
    ["mean", "count"]
)
banded.rename(columns={"mean": "approval_rate"}).round(3)

## 3. Feature selection

Three independent families, then a consensus:

- **filter** — mutual information, model-free
- **embedded** — LightGBM split gain, taken from the model's internals
- **wrapper** — permutation importance: the PR-AUC a fitted model *loses* when a column is
  shuffled, measured on rows it never saw

They disagree by construction, so their scores are rank-normalised onto a common scale and
averaged. The final set is then chosen by cross-validation, not by the ranking alone.

In [ ]:
train_features = train_frame[features]
train_target = train_frame[TARGET_COLUMN]

ranking = rank_features(train_features, train_target, "lightgbm")
ranking.round(4)

In [ ]:
selected, curve = select_features(train_features, train_target, ranking, "lightgbm")
print(f"selected {len(selected)} of {len(features)} features")
print(selected)
plot_selection_curve(curve, selected=len(selected))

### Would a learned embedding find more?

The engineered features encode the interaction by hand. An autoencoder asks the opposite
question: fit a bottleneck network to reconstruct the inputs, then let its compressed middle
layer stand in as learned features. If the network finds structure the hand-built features
missed, adding its embedding should lift the cross-validated score by more than one standard
error — and anything smaller is a tie the simpler model wins.

In [ ]:
from credit_risk.features.autoencoder import autoencoder_verdict

autoencoder_verdict(train_features, train_target, "lightgbm", numeric, categorical)

## 4. Model comparison

Every registered model, scored by stratified cross-validation on the training rows only.

In [ ]:
selected_numeric, selected_categorical = split_feature_types(selected)
comparison = pd.DataFrame(
    {
        name: cross_validated_score(
            train_features[selected],
            train_target,
            name,
            selected_numeric,
            selected_categorical,
        )
        for name in sorted(MODEL_BUILDERS)
    }
).T[["mean", "std"]]
comparison.sort_values("mean", ascending=False).round(4)

In [ ]:
plot_model_comparison(comparison)

## 5. Hyperparameter tuning

Optuna searches against cross-validated PR-AUC. Every trial is scored *inside* a
cross-validation loop, so the preprocessing is refit per fold and the holdout is never touched.

In [ ]:
study = tune_model(
    train_features[selected],
    train_target,
    "lightgbm",
    selected_numeric,
    selected_categorical,
    trials=CONFIG.training.optuna_trials,
)
print(f"best cross-validated PR-AUC: {study.best_value:.4f}")
print(f"best parameters: {study.best_params}")

In [ ]:
plot_tuning_history(study)

In [ ]:
plot_param_importances(study)

## 6. Final model and decision threshold

The threshold is a business decision, not a fit: a false approval costs more than a false
rejection, so it is chosen by minimising expected cost on out-of-fold training predictions.

In [ ]:
model = build_model("lightgbm", selected_numeric, selected_categorical, study.best_params)
model.fit(train_features[selected], train_target)

holdout_features = holdout_frame[selected]
holdout_target = holdout_frame[TARGET_COLUMN]
probabilities = model.predict_proba(holdout_features)[:, 1]

# In-sample probabilities are overconfident, so the threshold is chosen on out-of-fold
# predictions. The holdout stays untouched until the final score.
out_of_fold = out_of_fold_probabilities(
    train_features[selected],
    train_target,
    "lightgbm",
    selected_numeric,
    selected_categorical,
    study.best_params,
)
threshold = optimal_threshold(train_target, out_of_fold)
print(f"false approval costs {CONFIG.training.false_approval_cost}x a false rejection")
print(f"cost-optimal threshold: {threshold:.2f} (default would be 0.50)")

In [ ]:
calibrated = calibrate_model(
    build_model("lightgbm", selected_numeric, selected_categorical, study.best_params),
    train_features[selected],
    train_target,
)
metrics = score_model(model, holdout_features, holdout_target, threshold)
metrics["brier_score_calibrated"] = score_model(
    calibrated, holdout_features, holdout_target, threshold
)["brier_score"]
pd.Series(metrics).to_frame("holdout").round(4)

## 7. Model dynamics

In [ ]:
plot_learning_curve(
    train_features[selected],
    train_target,
    "lightgbm",
    selected_numeric,
    selected_categorical,
    study.best_params,
)

In [ ]:
plot_roc_and_pr_curves({"lightgbm": model}, holdout_features, holdout_target)

In [ ]:
plot_calibration_curve(
    {"lightgbm": model, "lightgbm (calibrated)": calibrated}, holdout_features, holdout_target
)

In [ ]:
plot_threshold_cost(holdout_target, probabilities, chosen=threshold)

In [ ]:
plot_confusion_matrix(model, holdout_features, holdout_target, threshold)

## 8. Explainability

The beeswarm shows which features matter and in which direction. The dependence plot is the
decisive one: colouring credit score by employment shows whether the two *interact* — the soft
AND the exploratory analysis predicted.

In [ ]:
explanation = explain_model(model, holdout_features)
global_importance(explanation).head(10).to_frame()

In [ ]:
plot_beeswarm(explanation)

In [ ]:
plot_importance_bar(explanation)

In [ ]:
influential = top_features(explanation, 2)
print(f"dependence on the two features the model leans on most: {influential}")
plot_dependence(explanation, influential[0], influential[1])

### Is the soft AND really there?

Selection dropped raw `CreditScore` — the engineered `credit_score_x_employed` already
carries it. To see the *interaction itself*, explain a model fitted on the original
columns only, where credit score and employment still exist separately and neither is
collinear with an engineered stand-in. If they interact, the two colours separate: the
same credit score earns a different contribution depending on employment.

In [ ]:
# CreditScore and credit_score_x_employed are collinear, so a model holding both
# splits the credit signal between them and the dependence plot becomes unreadable.
# Fit on the ORIGINAL columns only: then the interaction has nowhere to hide.
from credit_risk.data.schema import CATEGORICAL_FEATURES, NUMERIC_FEATURES

original = [
    column
    for column in NUMERIC_FEATURES + CATEGORICAL_FEATURES
    if column not in CONFIG.sensitive_features
]
original_numeric, original_categorical = split_feature_types(original)

raw_model = build_model("lightgbm", original_numeric, original_categorical)
raw_model.fit(train_frame[original], train_target)
raw_explanation = explain_model(raw_model, holdout_frame[original])

plot_dependence(raw_explanation, "CreditScore", "EmploymentType_Unemployed")

### A decision, a probability, and reasons a human can read

In [ ]:
decision_reasons(model, holdout_features.head(10), threshold)

In [ ]:
declined = decision_reasons(model, holdout_features, threshold)
worst = declined["probability"].idxmin()
local_contributions(explanation, holdout_features.index.get_loc(worst)).head(6).round(3)

## 9. Error analysis

Where the model is wrong, how wrong, and on whom.

In [ ]:
classified = classify_predictions(model, holdout_features, holdout_target, threshold)

# Selection dropped some raw columns from the model, but the error analysis still needs them
# to describe *who* the model fails on.
CONTEXT = ["CreditScore", "Income", "EmploymentType", "credit_band"]
for column in CONTEXT:
    classified[column] = holdout_frame[column]

error_summary(classified)

In [ ]:
plot_error_overview(classified)

In [ ]:
error_profile(classified, selected)

In [ ]:
error_rate_by_segment(classified, "EmploymentType")

In [ ]:
error_rate_by_segment(classified, "credit_band")

In [ ]:
plot_error_rate_by_segment(classified, ["EmploymentType", "credit_band"])

### The mistakes the model was surest about

In [ ]:
confident_mistakes(classified, count=8)[
    ["CreditScore", "Income", "EmploymentType", TARGET_COLUMN, "probability", "outcome"]
]

## 10. Fairness audit

`Gender` and `City` are excluded from the model on fair-lending grounds. That is *not* the same
as the model being fair: another column can act as a proxy and reintroduce the attribute through
the back door. The only way to know is to hold the decisions up against the groups and look.

The screen is the **four-fifths rule**: the least-selected group must reach 80% of the
most-selected one. Below that, a regulator asks the lender to justify itself.

In [ ]:
from credit_risk.fairness import (
    disparate_impact,
    fairness_audit,
    impact_ratio_confidence,
    plot_fairness,
)

for column in CONFIG.sensitive_features:
    classified[column] = holdout_frame[column]

audit = fairness_audit(classified)
audit

In [ ]:
disparate_impact(audit)

In [ ]:
plot_fairness(audit)

### Is the result conclusive?

A point estimate on a thousand applicants split across four cities cannot separate real
disparate impact from sampling noise. Bootstrapping the ratio says how much of the interval sits
below the threshold — bias can only be *concluded* when the whole interval does.

In [ ]:
import pandas as pd

pd.DataFrame(
    [
        {"attribute": column, **impact_ratio_confidence(classified, column)}
        for column in CONFIG.sensitive_features
    ]
).set_index("attribute")

## 11. Takeaways

In [ ]:
baseline = cross_validated_score(
    train_frame[features], train_target, "logistic_regression", numeric, categorical
)
errors = classified[classified["is_error"]]
false_approvals = int((classified["outcome"] == "false_positive").sum())
false_rejections = int((classified["outcome"] == "false_negative").sum())

print(f"baseline logistic regression (all features) : PR-AUC {baseline['mean']:.3f}")
print(f"tuned lightgbm on the selected set          : PR-AUC {metrics['average_precision']:.3f}")
print(f"features kept                               : {len(selected)} of {len(features)}")
print(f"top feature by SHAP                         : {global_importance(explanation).index[0]}")
print(f"decision threshold (cost-optimal)           : {threshold:.2f}")
print(
    f"calibration improved Brier                  : "
    f"{metrics['brier_score']:.4f} -> {metrics['brier_score_calibrated']:.4f}"
)
print(
    f"holdout errors                              : {len(errors)} "
    f"({false_approvals} false approvals, {false_rejections} false rejections)"
)

- **The engineered interaction beats the raw signal.** `credit_score_x_employed`
  outranks `CreditScore` itself and is the top feature by SHAP. Selection then drops raw
  credit score and employment entirely — the interaction already carries both.
- **Feature engineering closed most of the model gap.** A linear model on the engineered
  features scores close to the tuned gradient booster, where on the raw features it lagged
  badly. Encoding the soft AND explicitly is what a linear model cannot discover on its own.
- **The autoencoder found nothing left to find.** Its embedding moves the cross-validated
  score by less than one standard error, so it is measured, reported, and dropped. The
  hand-built interaction had already taken the signal the network was looking for.
- **Outlier removal is a measured trade.** The detectors disagree with each other and the
  flagged rows approve near the base rate, so on this sample removing them *lowers* the
  cross-validated score by about 0.003 — not a win here, where the flagged rows are extreme
  but plausible. The pipeline still removes them by default to exercise the full
  detect-and-remove path; `--keep-outliers` opts out. The impossible values — a domain rule,
  not a statistical one — are always repaired regardless.
- **Excluding a protected attribute is not fairness.** `Gender` and `City` never enter the
  model, yet the approval rates still differ by city enough to fail the four-fifths screen on
  the point estimate. The bootstrap says the holdout is too small to call it either way —
  which is the honest answer, and the reason the screen is reported rather than buried.
- **The threshold matters more than the last point of PR-AUC.** With false approvals priced
  above false rejections, the cost-optimal cut-off sits far from the default 0.5, and moving
  it changes the error mix far more than any hyperparameter did.